In [ ]:
# ==========================================
2 #Instruction-Tuning Dataset Preparation
3 # ==========================================
4 from flask import json
5 import pandas as pd
6 import gzip
7 import os
8 import random
9 import re
10 import nltk
11 from nltk.corpus import words
12 import torch
13 from torch.utils.data import Dataset, DataLoader
14
15 # -------------------------------
16 # 1. Setup
17 # -------------------------------
18 nltk.download('words')
19 english_words = set(words.words())
20 device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
21 print("Running on:", device)
22
23 # -------------------------------
24 # 2. Read JSONL Files
25 # -------------------------------
ef parse(path):
27 with gzip.open(path, 'rb') as g:
28 for line in g:
29 yield json.loads(line)
2
30
31 def getDF(path):
32 data = {}
33 for i, d in enumerate(parse(path)):
34 data[i] = d
35 return pd.DataFrame.from_dict(data, orient='index')
36
37 # Update file paths as needed
38 dfd = getDF(r'C:/Users/HP/Downloads/All_Beauty.jsonl.gz') # review data
39 dfm = getDF(r'C:/Users/HP/Downloads/meta_All_Beauty.jsonl.gz') # metadata
40
41 # -------------------------------
42 # 3. Drop Unnecessary Columns
43 # -------------------------------
44 # Drop columns only if they exist (for safety)
45 dfd.drop(columns=[c for c in [
46 'rating', 'title', 'text', 'images', 'timestamp',
47 'helpful_vote', 'verified_purchase'
48 ] if c in dfd.columns], inplace=True, errors='ignore')
49
50 dfm.drop(columns=[c for c in [
51 'main_category', 'average_rating', 'rating_number', 'features',
52 'description', 'price', 'images', 'videos', 'store',
53 'categories', 'details', 'bought_together'
54 ] if c in dfm.columns], inplace=True, errors='ignore')
55
56 # -------------------------------
57 # 4. Filter Active Users
58 # -------------------------------
59 reviewer_counts = dfd['user_id'].value_counts()
60 reviewer_ids_at_least_5 = reviewer_counts[reviewer_counts >= 5].index
61 dfd = dfd[dfd['user_id'].isin(reviewer_ids_at_least_5)]
62 dfd.reset_index(inplace=True, drop=True)
63
64 # -------------------------------
65 # 5. Merge Data on 'parent_asin'
66 # -------------------------------
67 df = pd.merge(dfd, dfm, on='parent_asin', how='inner')
68
69 # -------------------------------
70 # 6. Clean Product Titles
71 # -------------------------------
72 def clean_text(text):
73 if not isinstance(text, str):
74 text = str(text)
75 text = re.sub(r'[^\w\s\.,\?]|(?<=\s)\w(?=\s)', ' ', text) # remove special chars
76 text = re.sub(r'\b\w*[\d_]+\w*\b', '', text) # remove digits/underscores
77 text = re.sub(r'\bmisspelled\b|\bmeaningless\b|\bwordss\b|\bsymbols\b', ' ', text,
flags=re.IGNORECASE)


In [ ]:
text = ' '.join(word for word in text.split() if word.lower() in english_words) # keep
English words only
3
79 text = re.sub(r'\b\w\b', ' ', text) # remove single chars
80 text = re.sub(r'\s+', ' ', text) # normalize spaces
81 return text.strip()
82
83 if 'title' in df.columns:
84 df['title'] = df['title'].apply(clean_text)
85 else:
86 raise KeyError("No title column found after merging. Please check dfm columns.")
87
88 # -------------------------------
89 # 7. Group by User and Build Histories
90 # -------------------------------
91 grouped_df = df.groupby('user_id').apply(lambda x: {
92 'data': {
93 'total_asin': len(x),
94 **{
95 str(i + 1): x.iloc[i, 1:].to_dict() # store each item as dict
96 for i in range(len(x))
97 }
98 }
99 }).reset_index()
100
101 grouped_df.columns = ['ID', 'info']
102
103 # -------------------------------
104 # 8. Extract First 5 Item Titles
105 # -------------------------------
106 for i in range(1, 6):
107 grouped_df[f'item_{i}_title'] = grouped_df['info'].apply(
108 lambda x: x['data'].get(str(i), {}).get('title', '')
109 )
110
111 # -------------------------------
112 # 9. Create Prompts Sample and Outputs
113 # -------------------------------
114 grouped_df['prompts'] = (
115 "Reviewer has bought " +
116 grouped_df['item_1_title'] + ", " +
117 grouped_df['item_2_title'] + ", " +
118 grouped_df['item_3_title'] + ", " +
119 grouped_df['item_4_title'] +
120 ". Considering their purchase history, which product will they buy next from the below
options?"
121 )
122 grouped_df['outputs'] = grouped_df['item_5_title']
123
124 # -------------------------------
125 # 10. Add Negative Samples
126 # -------------------------------
127 def get_titles(asins):
128 titles = []
4
129 for asin in asins:
130 t = dfm.loc[dfm['parent_asin'] == asin, 'title'].iloc[0]
131 titles.append(t)
132 return titles
133
134 def create_additional_columns(row):
135 total_asin = row['info']['data']['total_asin']
136 used_asins = [row['info']['data'][str(i)]['parent_asin'] for i in range(1, total_asin +
1)]
137 available_asins = dfm[~dfm['parent_asin'].isin(used_asins)]['parent_asin'].tolist()
138 random.shuffle(available_asins)
139 random_items = random.sample(available_asins, 3)
140 return pd.Series(get_titles(random_items), index=['item_1_out', 'item_2_out',
'item_3_out'])
141
142 grouped_df[['item_1_out', 'item_2_out', 'item_3_out']] =
grouped_df.apply(create_additional_columns, axis=1)
143
144 # -------------------------------
145 # 11. Build Final Prompt Options
146 # -------------------------------
147 def generate_prompt(row):
148 items = [row['outputs'], row['item_1_out'], row['item_2_out'], row['item_3_out']]
149 random.shuffle(items)
150 return f" A) {items[0]}, B) {items[1]}, C) {items[2]}, D) {items[3]}."
151
152 grouped_df['prompts_part2'] = grouped_df.apply(generate_prompt, axis=1)
153 grouped_df['prompts'] = grouped_df['prompts'].str.cat(grouped_df['prompts_part2'], sep=' ')
154
155 # -------------------------------
156 # 12. Save Final Dataset
157 # -------------------------------
158 LLM_data = grouped_df[['prompts', 'outputs']]
159 print(LLM_data.head())
160
161 LLM_data.to_csv(r'C:/Users/HP/Downloads/final_dataset_sft.csv', index=False)
162 print(" Final dataset saved successfully!")